In [1]:
#!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['WorldWideTelescope/pywwt', 'chakravala/Grassmann.jl', 'roboticslibrary/rl', 'ROBelgium/MSNoise', 'stepjam/RLBench', 'FragIt/fragit-main', 'architecture-building-systems/revitpythonshell', 'VLSIDA/OpenRAM', 'kaczmarj/neurodocker', 'code-iai/ROSIntegration']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,000287679e1bef118cffcc087810c3fea90b6dfe,worldwidetelescope_pywwt


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  0%|▏                                                    | 4/1477 [00:04<25:15,  1.03s/it]

Got Errors {'058b65fbfffa0af1168fd0fe09465a82e3488d69': 'Key 058b65fbfffa0af1168fd0fe09465a82e3488d69 not found in /da5_fast/All.sha1c/commit_5.tch'}


  1%|▍                                                   | 11/1477 [00:11<24:56,  1.02s/it]

Got Errors {'0ea134e94428a4d551221e76c7f64b761b52af5e': 'Key 0ea134e94428a4d551221e76c7f64b761b52af5e not found in /da5_fast/All.sha1c/commit_14.tch'}


  1%|▋                                                   | 18/1477 [00:18<24:47,  1.02s/it]

...

Got Errors {'f2aa5c5bb2a504e4b2a457fa688bd35f57f536b0': 'Key f2aa5c5bb2a504e4b2a457fa688bd35f57f536b0 not found in /da5_fast/All.sha1c/commit_114.tch'}


 96%|████████████████████████████████████████████████  | 1418/1477 [24:22<01:01,  1.04s/it]

Got Errors {'f3e867154d91c63cf82557b2b4c354384f0aa367': 'Key f3e867154d91c63cf82557b2b4c354384f0aa367 not found in /da5_fast/All.sha1c/commit_115.tch'}


100%|██████████████████████████████████████████████████| 1477/1477 [25:24<00:00,  1.03s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,000287679e1bef118cffcc087810c3fea90b6dfe,a84e7e257a5846ce8ab8f25a4552e969f54682c4,[80fcf7f7ba373373cd8f07c923bf71adae751bf7],Peter Williams <peter@newton.cx>,1602898244,-0400,Peter Williams <peter@newton.cx>,1602898244,-0400,ci/azure-build-and-test.yml: skip Windows/Pyth...,000287679e1bef118cffcc087810c3fea90b6dfe,worldwidetelescope_pywwt
1,00160f35b811acb815ef8f83be5b8b08450699cc,7d95b8443f8388457efed560f10f9fb51f2c6352,[3ef4ec9a3dd615ebf4808cb8a3a468dc34a679e1],Peter Williams <peter@newton.cx>,1735501022,-0500,Peter Williams <peter@newton.cx>,1735502749,-0500,ci/azure-build-and-test.yml: update to macos-1...,00160f35b811acb815ef8f83be5b8b08450699cc,worldwidetelescope_pywwt


In [37]:
# fixed to use df_commit_data to fix project assignemnt error

# what does the first record looks like?
# commit  parents
v = df_commit_data.iloc[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

commit                     000287679e1bef118cffcc087810c3fea90b6dfe
tree                       a84e7e257a5846ce8ab8f25a4552e969f54682c4
parent                   [80fcf7f7ba373373cd8f07c923bf71adae751bf7]
author                             Peter Williams <peter@newton.cx>
author_time                                              1602898244
author_tz                                                     -0400
committer                          Peter Williams <peter@newton.cx>
committer_time                                           1602898244
committer_tz                                                  -0400
message           ci/azure-build-and-test.yml: skip Windows/Pyth...
sha1                       000287679e1bef118cffcc087810c3fea90b6dfe
project                                    worldwidetelescope_pywwt
Name: 0, dtype: object


In [38]:
# fixed to use df_commit_data to fix project assignemnt error

# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for i, k in df_commit_data.iterrows():
  row = pd.Series({'project':k['project'], 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [39]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,worldwidetelescope_pywwt,000287679e1bef118cffcc087810c3fea90b6dfe,Peter Williams <peter@newton.cx>,1602898244,ci/azure-build-and-test.yml: skip Windows/Pyth...


In [36]:
#mode a means append, so you have all your projects in the same file
yournetid='hneel'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

#### WorldWideTelescope/pywwt
- Stars: 55
- Forks: 14
- Last Commit Date: July 11th, 2026

#### chakravala/Grassmann.jl
- Stars: 515
- Forks: 41
- Last Commit Date: August 9th, 2026

#### roboticslibrary/rl
- Stars: 1.2k
- Forks: 244
- Last Commit Date: April 15th, 2025

#### ROBelgium/MSNoise
- Stars: 214
- Forks: 94
- Last Commit Date: August 20th, 2026

#### stepjam/RLBench
- Stars: 1.8k
- Forks: 323
- Last Commit Date: January 25th, 2025

#### FragIt/fragit-main
- Stars: 28
- Forks: 15
- Last Commit Date: April 11th, 2025

#### architecture-building-systems/revitpythonshell
- Stars: 595
- Forks: 125
- Last Commit Date: April 11th, 2025

#### VLSIDA/OpenRAM
- Stars: 1.1k
- Forks: 288
- Last Commit Date: August 16th, 2026

#### kaczmarj/neurodocker
- Stars: 361
- Forks: 99
- Last Commit Date: September 23rd, 2026 

#### code-iai/ROSIntegration
- Stars: 468
- Forks: 142
- Last Commit Date: November 3rd, 2025

In [56]:
for prj in projects:
  print(prj)
  prj = prj.lower().replace('/','_',2)
  mask = dfinf['project'] == prj
  temp = dfinf[mask]
  print("Commits:", len(temp))
  print("Authors:", temp['author'].nunique())
  print("Max Time:", temp['time'].max())
  print("Min Time:", temp['time'].min())
  print("")

WorldWideTelescope/pywwt
Commits: 1725
Authors: 16
Max Time: 1757020950
Min Time: 1389625907

chakravala/Grassmann.jl
Commits: 597
Authors: 15
Max Time: 1760912173
Min Time: 1503791400

roboticslibrary/rl
Commits: 578
Authors: 20
Max Time: 1750643853
Min Time: 1402513214

ROBelgium/MSNoise
Commits: 2084
Authors: 52
Max Time: 1776211363
Min Time: 1377438755

stepjam/RLBench
Commits: 619
Authors: 38
Max Time: 1756280121
Min Time: 1569533042

FragIt/fragit-main
Commits: 406
Authors: 5
Max Time: 1747942920
Min Time: 1335610324

architecture-building-systems/revitpythonshell
Commits: 656
Authors: 29
Max Time: 1744359920
Min Time: 1258533985

VLSIDA/OpenRAM
Commits: 5074
Authors: 129
Max Time: 1761069676
Min Time: 1478107108

kaczmarj/neurodocker
Commits: 2490
Authors: 87
Max Time: 1761955608
Min Time: 1487367191

code-iai/ROSIntegration
Commits: 467
Authors: 48
Max Time: 1762166372
Min Time: 1501619346

